#### Health Connect Model #### 
HealthConnect Clinic is facing high rates of missed appointments. The goal is to design a machine learning system that predicts whether a patient will attend or miss their scheduled appointment. This will enable proactive interventions such as reminders, rescheduling, or targeted support.  

We will begin by installing the necessary dependencies in our environment:

In [ ]:
%pip install numpy pandas scikit-learn matplotlib seaborn xgboost pyyaml

  Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl (8.2 MB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
    --------------------------------------- 0.5/36.6 MB 318.8 MB/s eta 0:00:01
    --------------------------------------- 0.5/36.6 MB 318.8 MB/s eta 0:00:01
   - -------------------------------------- 1.0/36.6 MB 1.6 MB/s eta 0:00:23
   - -------------------------------------- 1.6/36.6 MB 2.1 MB/s eta 0:00:17
   -- ------------------------------------- 2.1/36.6 MB 2.1 MB/s eta 0:00:17
   -- ------------------------------------- 2.6/36.6 MB 2.3 MB/s eta 0:00:16
   ---


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


It is also good practice to verify installation:

In [6]:
import numpy as np
import pandas as pd
import sklearn
import xgboost
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import os

print("All libraries loaded successfully!")

All libraries loaded successfully!


Now, unlike our previous project, where we wrote and defined all our functions within our notebook here step by step, we have chosen to adopt a more standard and structured way of processing, testing, and development.

We have built our entire pipeline within smaller components that we can easily call here and run, for cleanliness, and reusability. 

We have predefined a preprocessing.py components that contains all the functions to carry out our data preprocessing, a train.py that does the training on our selected models, and a an evaluate.py that carries out evaluation on each of the models and generates results for analysis and report.

Another reason for doing this is to have a well defined and arranged repo structure that can be easily maneuvered by visitors interested in exploring our models.

So naturally, the first step we will carry out is data preprocessing:

In [7]:
from src.preprocessing import preprocess_pipeline

df = preprocess_pipeline("data/HealthConnect_Appointment_Data.csv")

## Preprocessing Summary
**Built a robust preprocessing pipeline (preprocessing.py) that:**  

- Handles missing values with SimpleImputer (categorical → most frequent, numeric → median).

- Converts date columns into engineered features (lead_time_days, appointment_dayofweek, appointment_month).

- Drops raw date columns after feature engineering.

- Saves both the processed dataset and the pipeline object for reuse.

**Fixed earlier errors:**

- FileNotFoundError when saving pipeline → solved by creating the models/ directory with os.makedirs(..., exist_ok=True).

Our next step is training:

In [14]:
!python src/train.py

=== Logistic Regression Results ===
Accuracy: 0.527
Precision: 0.5217519872896806
Recall: 0.527
F1 Score: 0.524262848154458

Classification Report:
               precision    recall  f1-score   support

    Attended       0.54      0.55      0.54       463
   Cancelled       0.10      0.08      0.09        52
     No-Show       0.55      0.55      0.55       485

    accuracy                           0.53      1000
   macro avg       0.40      0.39      0.39      1000
weighted avg       0.52      0.53      0.52      1000


=== Random Forest Results ===
Accuracy: 0.576
Precision: 0.5460031064354455
Recall: 0.576
F1 Score: 0.5605973518302285

Classification Report:
               precision    recall  f1-score   support

    Attended       0.57      0.59      0.58       463
   Cancelled       0.00      0.00      0.00        52
     No-Show       0.59      0.62      0.60       485

    accuracy                           0.58      1000
   macro avg       0.38      0.40      0.39      1000

## Training Summary
**Updated train.py to:**

- Load the processed dataset instead of raw data (avoids missing engineered features like lead_time_days).

- Use stratified train/test split to ensure all classes are represented.

- Train and save both Logistic Regression and Random Forest models.

**Fixed earlier errors:**

- NaN values in Logistic Regression → solved by adding imputers in preprocessing.

- Multiclass metrics error (average='binary') → solved by using average="weighted" for precision, recall, and F1.

- UndefinedMetricWarning (precision ill‑defined for classes with no predictions) → solved by adding zero_division=0 to metric calls.

Last in this workflow, we run evaluate.py to load the saved models and carry out evaluation on them:

In [17]:
!python src/evaluate.py


=== Logistic Regression Evaluation ===
Accuracy: 0.527
Precision: 0.5217519872896806
Recall: 0.527
F1 Score: 0.524262848154458

Classification Report:
               precision    recall  f1-score   support

    Attended       0.54      0.55      0.54       463
   Cancelled       0.10      0.08      0.09        52
     No-Show       0.55      0.55      0.55       485

    accuracy                           0.53      1000
   macro avg       0.40      0.39      0.39      1000
weighted avg       0.52      0.53      0.52      1000


=== Random Forest Evaluation ===
Accuracy: 0.576
Precision: 0.5460031064354455
Recall: 0.576
F1 Score: 0.5605973518302285

Classification Report:
               precision    recall  f1-score   support

    Attended       0.57      0.59      0.58       463
   Cancelled       0.00      0.00      0.00        52
     No-Show       0.59      0.62      0.60       485

    accuracy                           0.58      1000
   macro avg       0.38      0.40      0.39   

## Evaluation Summary

**Created evaluate.py to:**

- Reload saved models (logistic_regression.pkl, random_forest.pkl).

- Evaluate them on the test set using consistent metrics.

- Print accuracy, precision, recall, F1, and classification report with multiclass support.

**Generate confusion matrices for each model:**

- Saved automatically as PNG files for reproducibility.

- Displayed inline when running in a Jupyter Notebook for immediate visual inspection.

**Fixed earlier overlap with train.py by clarifying purpose:**

- train.py → fits and saves models.

- evaluate.py → reloads and evaluates models only.

## Key Errors Encountered & Fixes
- FileNotFoundError when saving pipeline → fixed with os.makedirs.

- NaN values in Logistic Regression → fixed with imputers.

- Missing engineered columns (lead_time_days) → fixed by training on processed dataset.

- Multiclass metrics error → fixed with average="weighted".

- UndefinedMetricWarning → fixed with zero_division=0 and stratified splitting.